# Download the Rain Forecast from CDS

As we are not using an environment for the project, we are having issues installing earthkit. 
After the instalation (below), it is necessary to uninstall and reinstall cftime. like so:

```
pip install "numpy<=2.0" cftime netCDF4 --force-reinstall --no-cache-dir --break-system-packages
```

In [1]:
%load_ext autoreload
%autoreload 2

import importlib.util

if importlib.util.find_spec("earthkit") is None:
    %pip install earthkit --break-system-packages --quiet --ignore-installed
else:
    print("earthkit is already installed")

earthkit is already installed


## Download Forecasts - Earthkit

Use form `https://cds.climate.copernicus.eu/datasets/seasonal-monthly-single-levels?tab=download` to check request attributes.

In [2]:
import earthkit.data as ekd
import rioxarray  # noqa: F401 -- register the spatial xarray accessor

In [3]:
YEAR = "2026"
START_MONTH = "11"
# we are specifying the lead time months for the forecast (5 months from the start month)
LEADTIME_MONTHS = ["1", "2", "3", "4", "5"]

In [4]:
# Specify the models to be used for the forecast
MODELS = {
    "UKMO GloaSea6-GC5.1": {"originating_centre": "ukmo", "system": "610"},
    "ECMWF SEAS5": {"originating_centre": "ecmwf", "system": "51"},
    "Meteo France S9": {"originating_centre": "meteo_france", "system": "9"},
    "DWD GCFS2.2": {"originating_centre": "dwd", "system": "22"},
    "CMCC SPS4": {"originating_centre": "cmcc", "system": "4"},
    "NCEP CFSv2": {"originating_centre": "ncep", "system": "2"},
    "JMA CPS4": {"originating_centre": "jma", "system": "4"},
    "ECCC GEM5.2-NEMO": {"originating_centre": "eccc", "system": "5"},
    "BOM ACCESS-S2": {"originating_centre": "bom", "system": "2"},
}

# Specify the datasets to be downloaded (precipitation, anomaly and hindcast mean)
DATASETS = {
    "prec": {
        "request": {"variable": "total_precipitation", "product_type": "ensemble_mean"},
        "source": "seasonal-monthly-single-levels",
    },
    "anomaly": {
        "request": {
            "variable": "total_precipitation_anomalous_rate_of_accumulation",
            "product_type": "ensemble_mean",
        },
        "source": "seasonal-postprocessed-single-levels",
    },
    "hindcast": {
        "request": {"variable": "total_precipitation", "product_type": "hindcast_climate_mean"},
        "source": "seasonal-monthly-single-levels",
    },
}

request = {
    "year": [YEAR],
    "month": [START_MONTH],
    "leadtime_month": LEADTIME_MONTHS,
    "area": [15, -90, -60, -30],  # Brazil bbox: North, West, South, East
    "data_format": "grib",
}


tprate_conversion = 1000 * 30 * 24 * 60 * 60  # convert from m.s-1 to mm/month

for model_name, model_request in MODELS.items():
    request.update(model_request)

    for dataset_name, dataset_metadata in DATASETS.items():
        request.update(dataset_metadata["request"])
        gribdata = ekd.from_source(
            "cds",
            dataset_metadata["source"],
            request=request,
        )

        array = gribdata.to_xarray() * tprate_conversion
        array = array.rio.write_crs("epsg:4326")

        model_shortname = model_request["originating_centre"]
        path = f"/data/CN3S/forecasts/{dataset_name}/{model_shortname}_{YEAR}{START_MONTH}.nc2"

        print(f"Saving {model_name}: {path}")
        array.to_netcdf(path, engine="netcdf4")

2026-11-16 20:50:26,380 INFO [2026-10-06T00:00:00Z] **Data Stores Service protection measures** affecting data downloads and account access: [please read our Announcement](https://forum.ecmwf.int/t/data-stores-service-protection-measures-affecting-data-downloads-and-account-access-cds-ads-ewds-ecds/15481) before enquiring with ECMWF Support.


2026-11-16 20:50:27,596 INFO Request ID is 0e3dee3a-61f7-439d-9143-32df3ba4b9bf


2026-11-16 20:50:27,832 INFO status has been updated to accepted


2026-11-16 20:50:49,988 INFO status has been updated to running


2026-11-16 20:51:01,609 INFO status has been updated to successful


f7bb02688ca7851c534a8e788fd62f00.grib:   0%|          | 0.00/44.6k [00:00<?, ?B/s]

... (173 outputs left out)

2026-11-16 21:09:42,225 INFO status has been updated to running


2026-11-16 21:09:59,516 INFO status has been updated to successful


55040d53153769f811a8e8a78a1d2dfa.grib:   0%|          | 0.00/44.6k [00:00<?, ?B/s]

Saving BOM ACCESS-S2: /data/CN3S/forecasts/hindcast/bom_202611.nc2
